In [1]:
import pandas as pd
import json

from medical_rag.config import MEDICAL_PATTERNS, MEDICAL_SYNONYMS
from medical_rag.chroma_indexer import PubMedChromaIndexer
from medical_rag.query_processor import MedicalQueryProcessor
from medical_rag.retriever import MedicalChromaRetriever
from medical_rag.query_services import (
    enhance_medical_query,
    print_enhanced_query
)
from medical_rag.mesh_terminology import (
    MeshLexicon,
    translate_known_medical_terms,
)

e:\anaconda3\envs\medrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化医学查询检索器

In [2]:
#加载已知期刊列表
chunks_df = pd.read_parquet(
    "F:/RAG/data/pubmed_fulltext_chunks_2.parquet"
)

journal_series = (
    chunks_df["journal"]
    .fillna("")
    .astype(str)
    .str.strip()
)

journal_series = journal_series[
    journal_series.ne("")
]

journal_counts = (
    journal_series
    .value_counts()
)

canonical_journal_map = {}

# value_counts 已按频率从高到低排序
for journal_name in journal_counts.index:
    normalized = journal_name.casefold()

    if normalized not in canonical_journal_map:
        canonical_journal_map[
            normalized
        ] = journal_name

known_journals = list(
    canonical_journal_map.values()
)


In [3]:
#初始化查询处理器与Retriever

mesh_lexicon = MeshLexicon(
    alias_index_path=r"F:\RAG\data\MedicalTerminology\mesh\mesh_alias_index.json",
    concepts_path=r"F:\RAG\data\MedicalTerminology\mesh\mesh_concepts.jsonl",
)

query_processor = (
    MedicalQueryProcessor(
        synonyms=MEDICAL_SYNONYMS,
        patterns=MEDICAL_PATTERNS,
        corpus_language="en",
        mesh_lexicon=mesh_lexicon
    )
)

indexer = PubMedChromaIndexer(
    model_name=(
        "BAAI/bge-small-en-v1.5"
    ),
    persist_directory=(
        r"F:\RAG\vector_db\chroma_pubmed_db_rebuilt"
    ),
    collection_name=(
        "pubmed_fulltext_bge_small"
    ),
    device="cuda",
    embedding_batch_size=32,
    insert_batch_size=1000
)

indexer.collection = (
    indexer.client.get_collection(
        name=indexer.collection_name
    )
)

print(
    "Indexed vectors:",
    indexer.collection.count()
)

Loaded 247,002 MeSH aliases
Loaded 31,110 MeSH concepts
Loading embedding model: BAAI/bge-small-en-v1.5
Device: cuda
Persist directory: F:\RAG\vector_db\chroma_pubmed_db_rebuilt
Collection name: pubmed_fulltext_bge_small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3015.18it/s]
F:\RAG\src\medical_rag\chroma_indexer.py:109: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.embedding_model.get_sentence_embedding_dimension()


Existing collections: ['pubmed_fulltext_bge_small']
Collection loaded successfully. Count: 631,000
Embedding dimension: 384
Distance metric: cosine
Indexed vectors: 631000


In [4]:
retriever = MedicalChromaRetriever(
    query_processor=query_processor,
    indexer=indexer,
    known_journals=known_journals,
    query_translator=translate_known_medical_terms
)

## 中文医学查询测试

In [5]:
query = (
    "检索近5年Nature communications中"
    "二甲双胍对心血管疾病结果的影响"
)

query_info = enhance_medical_query(
    query=query,
    query_processor=query_processor,
    retriever=retriever,
    indexer=indexer,
)

print_enhanced_query(query_info)

1. 原始查询
检索近5年Nature communications中二甲双胍对心血管疾病结果的影响

2. 基础清洗结果
检索近5年Nature communications中二甲双胍对心血管疾病结果的影响

3. 检测语言
zh

4. 识别医学实体
- 类型: drug, 文本: 二甲双胍, 位置: [27, 31]
- 类型: disease, 文本: 心血管疾病, 位置: [32, 37]

5. 医学同义词扩展
- 二甲双胍: ['metformin', 'dimethylbiguanide']
- 心血管疾病: ['cardiovascular disease', 'CVD']
- nature: ['D019368']

6. 向量检索查询
检索in the last five yearsNature communications中metformin对cardiovascular disease结果的影响 Related medical concepts: metformin; dimethylbiguanide; cardiovascular disease; CVD; D019368.

7. BGE完整查询（仅用于检查）
Represent this sentence for searching relevant passages: 检索in the last five yearsNature communications中metformin对cardiovascular disease结果的影响 Related medical concepts: metformin; dimethylbiguanide; cardiovascular disease; CVD; D019368.

8. 关键词检索查询
二甲双胍 OR 心血管疾病 OR metformin OR dimethylbiguanide OR "cardiovascular disease" OR CVD OR nature OR D019368

9. 关键词列表
['二甲双胍', '心血管疾病', 'metformin', 'dimethylbiguanide', 'cardiovascular disease', 'CVD', 'nature', 'D019368']

10

In [6]:
print("Collection:", indexer.collection_name)
print("Collection count:", indexer.collection.count())

print("\nVector query:")
print(query_info["vector_query"])

print("\nExtracted filters:")
print(
    json.dumps(
        query_info["extracted_filters"],
        indent=2,
        ensure_ascii=False,
    )
)

print("\nChroma where filter:")
print(
    json.dumps(
        query_info["where_filter"],
        indent=2,
        ensure_ascii=False,
    )
)

Collection: pubmed_fulltext_bge_small
Collection count: 631000

Vector query:
检索in the last five yearsNature communications中metformin对cardiovascular disease结果的影响 Related medical concepts: metformin; dimethylbiguanide; cardiovascular disease; CVD; D019368.

Extracted filters:
{
  "start_year": 2022,
  "end_year": 2026,
  "journal": "Nature Communications"
}

Chroma where filter:
{
  "$and": [
    {
      "journal": "Nature Communications"
    },
    {
      "publication_year": {
        "$gte": 2022
      }
    },
    {
      "publication_year": {
        "$lte": 2026
      }
    }
  ]
}


## 使用增强查询执行向量检索

In [7]:
search_output = retriever.search(
    query=query,
    n_results=10,
    candidate_multiplier=5,
    max_chunks_per_doc=2,
    diversify_documents=True,
)

print(search_output["results"])

   rank                    vector_id  distance  similarity  \
0     1     PMID_36030260_chunk_0001  0.329611    0.670389   
1     2     PMID_35803927_chunk_0000  0.333523    0.666477   
2     3     PMID_35803927_chunk_0037  0.333807    0.666193   
3     4     PMID_36030260_chunk_0000  0.334666    0.665334   
4     5  DOC_979625960f1d_chunk_0020  0.344816    0.655184   
5     6  DOC_327ad2b049a6_chunk_0002  0.344971    0.655029   
6     7  DOC_979625960f1d_chunk_0010  0.346930    0.653070   
7     8  DOC_992bd9468b2d_chunk_0055  0.347402    0.652598   
8     9  DOC_992bd9468b2d_chunk_0042  0.355185    0.644815   
9    10  DOC_327ad2b049a6_chunk_0030  0.355188    0.644812   

                                                text      pmid  \
0  (i) epigenetic changes in diabetic vascular di...  36030260   
1  Title: Cardiac disruption of SDHAF4-mediated m...  35803927   
2  Title: Cardiac disruption of SDHAF4-mediated m...  35803927   
3  Title: Reversal of the renal hyperglycemic mem... 

In [8]:
#查看查询增强信息

print(
    "原始查询：",
    search_output["original_query"]
)

print(
    "向量查询：",
    search_output["vector_query"]
)

print(
    "关键词查询：",
    search_output["keyword_query"]
)

print(
    "识别实体：",
    search_output["entities"]
)

print(
    "同义词扩展：",
    search_output[
        "synonym_expansions"
    ]
)

print(
    "过滤条件：",
    search_output["where_filter"]
)

print(
    "警告：",
    search_output["warnings"]
)

原始查询： 检索近5年Nature communications中二甲双胍对心血管疾病结果的影响
向量查询： 检索in the last five yearsNature communications中metformin对cardiovascular disease结果的影响 Related medical concepts: metformin; dimethylbiguanide; cardiovascular disease; CVD; D019368.
关键词查询： 二甲双胍 OR 心血管疾病 OR metformin OR dimethylbiguanide OR "cardiovascular disease" OR CVD OR nature OR D019368
识别实体： [{'entity_type': 'drug', 'text': '二甲双胍', 'start': 27, 'end': 31}, {'entity_type': 'disease', 'text': '心血管疾病', 'start': 32, 'end': 37}]
同义词扩展： {'二甲双胍': ['metformin', 'dimethylbiguanide'], '心血管疾病': ['cardiovascular disease', 'CVD'], 'nature': ['D019368']}
过滤条件： {'$and': [{'journal': 'Nature Communications'}, {'publication_year': {'$gte': 2022}}, {'publication_year': {'$lte': 2026}}]}
警告： ['查询包含中文，但当前BGE模型和PMC语料主要为英文；建议在向量检索前将完整查询翻译为英文。']


In [9]:
#查询结果

results_df = search_output["results"]

display_columns = [
    "rank",
    "similarity",
    "source_title",
    "journal",
    "publication_year",
    "pmid",
    "doc_id",
    "chunk_index",
    "text"
]

available_columns = [
    column
    for column in display_columns
    if column in results_df.columns
]

results_df[available_columns]

,rank,similarity,source_title,journal,publication_year,pmid,doc_id,chunk_index,text
0,1,0.670389,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260,PMID_36030260,1,(i) epigenetic changes in diabetic vascular di...
1,2,0.666477,Cardiac disruption of SDHAF4-mediated mitochon...,Nature Communications,2022,35803927,PMID_35803927,0,Title: Cardiac disruption of SDHAF4-mediated m...
2,3,0.666193,Cardiac disruption of SDHAF4-mediated mitochon...,Nature Communications,2022,35803927,PMID_35803927,37,Title: Cardiac disruption of SDHAF4-mediated m...
3,4,0.665334,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260,PMID_36030260,0,Title: Reversal of the renal hyperglycemic mem...
4,5,0.655184,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,,DOC_979625960f1d,20,Title: Cutaneous and acral melanoma cross-OMIC...
5,6,0.655029,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,2,Title: A Pilot randomized trial to examine eff...
6,7,0.653070,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,,DOC_979625960f1d,10,ment analysis of hyper- and hypomethylated DMR...
7,8,0.652598,The cholesterol uptake regulator PCSK9 promote...,Nature Communications,2022,,DOC_992bd9468b2d,55,Title: The cholesterol uptake regulator PCSK9 ...
8,9,0.644815,The cholesterol uptake regulator PCSK9 promote...,Nature Communications,2022,,DOC_992bd9468b2d,42,"PCSK9 to cholesterol biosynthesis, GGPS1-GGPP ..."
9,10,0.644812,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,30,Title: A Pilot randomized trial to examine eff...


## 对照测试多个查询

In [10]:
test_queries = [
    (
        "What are the effects of metformin "
        "on myocardial infarction?"
    ),
    (
        "近5年二甲双胍对心血管疾病"
        "结局有何影响？"
    ),
    (
        "检索2021年以来Nature Communications中"
        "关于acute kidney injury的研究"
    ),
    (
        "Does aspirin reduce mortality after MI?"
    ),
]

test_results = []

for test_query in test_queries:
    try:
        result = enhance_medical_query(
            query=test_query,
            query_processor=query_processor,
            retriever=retriever,
            indexer=indexer,
        )

        test_results.append({
            "query": test_query,
            "status": "PASS",
            "language":
                result["detected_language"],
            "entity_count":
                len(result["entities"]),
            "expansion_count":
                len(
                    result[
                        "synonym_expansions"
                    ]
                ),
            "vector_query":
                result["vector_query"],
            "keyword_query":
                result["keyword_query"],
            "where_filter":
                result["where_filter"],
        })

    except Exception as error:
        test_results.append({
            "query": test_query,
            "status": "FAIL",
            "error": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
        })

test_results_df = pd.DataFrame(test_results)

display(test_results_df)

,query,status,language,entity_count,expansion_count,vector_query,keyword_query,where_filter
0,What are the effects of metformin on myocardia...,PASS,en,2,3,What are the effects of metformin on myocardia...,"metformin OR ""myocardial infarction"" OR dimeth...",None
1,近5年二甲双胍对心血管疾病结局有何影响？,PASS,zh,2,2,in the last five yearsmetformin对cardiovascular...,二甲双胍 OR 心血管疾病 OR metformin OR dimethylbiguanid...,{'$and': [{'publication_year': {'$gte': 2022}}...
2,检索2021年以来Nature Communications中关于acute kidney ...,PASS,zh,0,4,检索2021年以来Nature Communications中关于acute kidney ...,"""acute kidney injury"" OR D058186 OR injury OR ...",{'$and': [{'journal': 'Nature Communications'}...
3,Does aspirin reduce mortality after MI?,PASS,en,3,4,Does aspirin reduce mortality after MI? Relate...,"aspirin OR mortality OR MI OR ""acetylsalicylic...",None
